# Test Preprocessing Lokal

Notebook di `src/finetuned/notebooks/` — path disesuaikan ke root project.

**Tujuan**: Validasi `preprocess_dataset()` tanpa GPU/Colab/zip.

## 1. Setup Path

In [1]:
import sys, os

# Notebook ada di src/finetuned/notebooks/ — naik 3 level ke root
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f'CWD: {os.getcwd()}')
print(f'ROOT in sys.path: {ROOT in sys.path}')

for f in [
    'src/finetuned/training/domain_trainer.py',
    'src/finetuned/data/dataset_loader.py',
    'dataset_aqg/output_domain/train.jsonl',
    'dataset_aqg/output_domain/validation.jsonl',
]:
    print(f'{'✅' if os.path.exists(f) else '❌'} {f}')

CWD: d:\2-Project\AQG
ROOT in sys.path: True
✅ src/finetuned/training/domain_trainer.py
✅ src/finetuned/data/dataset_loader.py
✅ dataset_aqg/output_domain/train.jsonl
✅ dataset_aqg/output_domain/validation.jsonl


## 2. Load Dataset

In [2]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()
DOMAIN_DIR = 'dataset_aqg/output_domain/'

train_dataset = loader.load_dataset(DOMAIN_DIR, split='train')
val_dataset   = loader.load_dataset(DOMAIN_DIR, split='validation')

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'Columns: {train_dataset.column_names}')

✓ Loaded 271 entries from dataset_aqg\output_domain\train.jsonl
✓ Loaded 33 entries from dataset_aqg\output_domain\validation.jsonl
Train: 271 | Val: 33
Columns: ['input', 'target', 'metadata']


In [3]:
# Drop metadata + deduplikasi
train_dataset = train_dataset.remove_columns(['metadata'])
val_dataset   = val_dataset.remove_columns(['metadata'])

seen, unique_idx = set(), []
for i, item in enumerate(train_dataset):
    if item['input'] not in seen:
        seen.add(item['input'])
        unique_idx.append(i)
train_dataset = train_dataset.select(unique_idx)

print(f'After dedup - Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'Columns: {train_dataset.column_names}')

After dedup - Train: 253 | Val: 33
Columns: ['input', 'target']


## 3. Load Tokenizer

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('LazarusNLP/IndoNanoT5-base')
print(f'pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})')
print(f'eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})')

pad_token: '<pad>' (id=0)
eos_token: '</s>' (id=1)


## 4. Test preprocess_dataset()

In [5]:
from src.finetuned.training.domain_trainer import DomainAdaptationTrainer
import torch

class StubModel:
    def parameters(self): return iter([torch.zeros(1)])
    def cuda(self): return self
    def enable_input_require_grads(self): pass

trainer = DomainAdaptationTrainer(
    model=StubModel(),
    tokenizer=tokenizer,
    output_dir='./test_output',
    max_length=512
)
print('Trainer initialized')

Trainer initialized


In [6]:
# Test train
print(f'Input: {len(train_dataset)} samples')
train_tok = trainer.preprocess_dataset(train_dataset)
print(f'Output: {len(train_tok)} samples')
print('✅ PASS' if len(train_tok) == len(train_dataset) else f'❌ FAIL: got {len(train_tok)}')

Input: 253 samples
Preprocessing 253 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing: 100%|##########| 253/253 [00:00<?, ? examples/s]

✓ Preprocessed 253 samples
  Sample label check: 217 valid tokens, 0 masked (-100)
Output: 253 samples
✅ PASS


In [7]:
# Test val
print(f'Input: {len(val_dataset)} samples')
val_tok = trainer.preprocess_dataset(val_dataset)
print(f'Output: {len(val_tok)} samples')
print('✅ PASS' if len(val_tok) == len(val_dataset) else f'❌ FAIL: got {len(val_tok)}')

Input: 33 samples
Preprocessing 33 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing: 100%|##########| 33/33 [00:00<?, ? examples/s]

✓ Preprocessed 33 samples
  Sample label check: 24 valid tokens, 0 masked (-100)
Output: 33 samples
✅ PASS


## 5. Inspect Sample

In [8]:
sample_raw = train_dataset[0]
sample_tok = train_tok[0]

print('=== RAW ===')
print(f'Input:  {sample_raw["input"][:80]}')
print(f'Target: {sample_raw["target"][:80]}')

print('\n=== TOKENIZED ===')
print(f'Keys: {list(sample_tok.keys())}')
print(f'input_ids len: {len(sample_tok["input_ids"])}')
print(f'labels len:    {len(sample_tok["labels"])}')

decoded = tokenizer.decode(sample_tok['input_ids'], skip_special_tokens=False)
print(f'\nDecoded input: {decoded[:100]}')
print('✅ Prefix OK' if decoded.startswith('question:') else '❌ Prefix MISSING')

=== RAW ===
Input:  Apa itu Object (Objek) dalam Python?
Target: | Nama | Deskripsi | Contoh |
|------|-----------|--------|
| **Class (Kelas)** 

=== TOKENIZED ===
Keys: ['input_ids', 'attention_mask', 'labels']
input_ids len: 20
labels len:    217

Decoded input: question: apa itu object (objek) dalam python?</s>
✅ Prefix OK


In [9]:
labels = sample_tok['labels']
n_valid  = sum(1 for l in labels if l != -100)
n_pad    = sum(1 for l in labels if l == tokenizer.pad_token_id)

print(f'Total labels:  {len(labels)}')
print(f'Valid tokens:  {n_valid}')
print(f'Pad remaining: {n_pad}')
print('✅ No pad in labels' if n_pad == 0 else f'⚠️ {n_pad} pad tokens still in labels')

valid_labels = [l for l in labels if l != -100]
print(f'\nDecoded target: {tokenizer.decode(valid_labels, skip_special_tokens=True)[:100]}')
print(f'Original target: {sample_raw["target"][:100]}')

Total labels:  217
Valid tokens:  217
Pad remaining: 0
✅ No pad in labels

Decoded target: | nama | deskripsi | contoh | |------|-----------|--------| | **class (kelas)** | cetakan (blueprint
Original target: | Nama | Deskripsi | Contoh |
|------|-----------|--------|
| **Class (Kelas)** | Cetakan (blueprint


## 6. Dummy Dataset (isolasi bug)

In [10]:
from datasets import Dataset

dummy_ds = Dataset.from_list([
    {"input": f"Apa itu konsep {i}?", "target": f"Konsep {i} adalah penting."}
    for i in range(10)
])
dummy_tok = trainer.preprocess_dataset(dummy_ds)
print(f'Dummy: {len(dummy_ds)} → {len(dummy_tok)}')
print('✅ PASS' if len(dummy_tok) == len(dummy_ds) else '❌ FAIL: bug di code, bukan dataset')

Preprocessing 10 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing:   0%|          | 0/10 [00:00<?, ? examples/s]

✓ Preprocessed 10 samples
  Sample label check: 7 valid tokens, 0 masked (-100)
Dummy: 10 → 10
✅ PASS


## 7. Summary

In [11]:
print('=' * 55)
print('PREPROCESSING VALIDATION SUMMARY')
print('=' * 55)

checks = [
    ('Train size preserved (253)',  len(train_tok) == len(train_dataset)),
    ('Val size preserved (33)',     len(val_tok)   == len(val_dataset)),
    ('Dummy size preserved (10)',   len(dummy_tok) == len(dummy_ds)),
    ('Task prefix ada',             tokenizer.decode(train_tok[0]['input_ids']).startswith('question:')),
    ('No pad_token in labels',      sum(1 for l in train_tok[0]['labels'] if l == tokenizer.pad_token_id) == 0),
    ('Labels punya valid tokens',   sum(1 for l in train_tok[0]['labels'] if l != -100) > 0),
]

all_pass = True
for name, result in checks:
    print(f'  {"✅ PASS" if result else "❌ FAIL"}: {name}')
    if not result: all_pass = False

print()
if all_pass:
    print('🎉 SEMUA PASS — Lanjut ke Section 8 untuk training sanity check.')
else:
    print('🚨 ADA YANG GAGAL — Fix dulu sebelum lanjut!')

PREPROCESSING VALIDATION SUMMARY
  ✅ PASS: Train size preserved (253)
  ✅ PASS: Val size preserved (33)
  ✅ PASS: Dummy size preserved (10)
  ✅ PASS: Task prefix ada
  ✅ PASS: No pad_token in labels
  ✅ PASS: Labels punya valid tokens

🎉 SEMUA PASS — Lanjut ke Section 8 untuk training sanity check.


## 8. Training Sanity Check (CPU, 2 steps)

Tujuan: confirm loss turun dari ~39 ke angka normal (~3-4) dengan fix yang sudah diterapkan.

Hanya 2 training steps — selesai dalam 2-3 menit di CPU.

In [12]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model

print('Loading model (CPU)...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    'LazarusNLP/IndoNanoT5-base',
    device_map='cpu'
)

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['q', 'v'],
    bias='none', task_type='SEQ_2_SEQ_LM'
)
peft_model = get_peft_model(base_model, lora_config)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f'✅ Model loaded | Trainable params: {trainable:,}')

Loading model (CPU)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Model loaded | Trainable params: 884,736


In [14]:
from src.finetuned.training.domain_trainer import DomainAdaptationTrainer

# Gunakan subset kecil — 8 samples saja untuk sanity check
train_small = train_dataset.select(range(8))
val_small   = val_dataset.select(range(4))

sanity_trainer = DomainAdaptationTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    output_dir='./test_output/sanity',
    max_length=128
)

# Training args untuk CPU Windows — num_workers=0 wajib di Windows
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir='./test_output/sanity',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    warmup_steps=0,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=1,
    fp16=False,
    predict_with_generate=False,
    load_best_model_at_end=False,
    report_to=['none'],
    dataloader_num_workers=0,      # ← WAJIB di Windows
    dataloader_pin_memory=False,   # ← Disable untuk CPU
    gradient_checkpointing=False,  # ← Disable untuk CPU sanity check
)

print('Starting sanity training on CPU...')
print('Expected loss: ~3-5 (normal), NOT ~39 (broken)')
print()

results = sanity_trainer.train(
    train_dataset=train_small,
    eval_dataset=val_small,
    training_args=training_args,
    early_stopping=False
)

final_loss = results['training_loss']
print(f'\nFinal loss: {final_loss:.4f}')

if final_loss < 10:
    print('✅ PASS: Loss normal! Fix bekerja.')
    print('   Aman untuk update zip dan training penuh di Colab.')
else:
    print(f'❌ FAIL: Loss masih tinggi ({final_loss:.1f}).')


Starting sanity training on CPU...
Expected loss: ~3-5 (normal), NOT ~39 (broken)


STARTING DOMAIN ADAPTATION TRAINING

⚠ GPU not available, training on CPU (will be slow)
  Model device: cpu
Preprocessing datasets...
Preprocessing 8 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing:   0%|          | 0/8 [00:00<?, ? examples/s]

✓ Preprocessed 8 samples
  Sample label check: 128 valid tokens, 0 masked (-100)
Preprocessing 4 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing:   0%|          | 0/4 [00:00<?, ? examples/s]

✓ Preprocessed 4 samples
  Sample label check: 24 valid tokens, 0 masked (-100)

=== Dataset Size After Preprocessing ===
Train samples (actual): 8
Eval samples (actual):  4
⚠ WARNING: Train dataset sangat kecil (8 sampel)!
  Kemungkinan ada bug di preprocessing. Cek kolom dataset.

=== Training Configuration ===
Epochs: 1
Batch size: 2
Gradient accumulation: 1
Effective batch size: 2
Learning rate: 0.0002
Warmup steps: 0
FP16: False
Train samples: 8
Eval samples: 4
Starting training...


Epoch,Training Loss,Validation Loss
1,7.237872,9.049692



=== Training Complete ===
Final training loss: 8.6091
Training time: 10.55 seconds
Training samples per second: 0.76
✓ Training results saved to test_output\sanity\training_results.json

Final loss: 8.6091
✅ PASS: Loss normal! Fix bekerja.
   Aman untuk update zip dan training penuh di Colab.
